# 4.2 Ekstraksi Fitur Menggunakan TSFEL

Notebook ini memuat implementasi kode Python untuk mengekstrak **68 fitur deret waktu terstandar** menggunakan pustaka **TSFEL** (*Time Series Feature Extraction Library*), dijalankan untuk **tiga parameter polutan**: **CO**, **NO₂**, dan **SO₂** di **Kecamatan Jabon, Kabupaten Sidoarjo**, sesuai ketentuan Tugas 3.

---

## 1. Spesifikasi Teknis Ekstraksi Fitur (Tugas 3)

* **Parameter Target**: **CO**, **NO₂** (Nitrogen Dioksida), **SO₂** (Sulfur Dioksida)
* **Total Fitur yang Diminta**: **Tepat 68 Fitur Kunci** per parameter
* **Frekuensi Sampling ($fs$)**: $1\text{ sampel/hari}$ ($fs = 1$)
* **Output Berkas**: `CO_Jabon_TSFEL.csv`, `NO2_Jabon_TSFEL.csv`, `SO2_Jabon_TSFEL.csv`, serta gabungan `ALL_Jabon_TSFEL.csv`


## 2. Import Library

In [1]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features


## 3. Konfigurasi Polutan

Peta nama polutan ke path file CSV masing-masing. Setiap file diasumsikan berada di
`data/downloads/jabon_pollutants_data/` dengan pola nama `jabon_<POLUTAN>.csv` dan memiliki
kolom `date` serta kolom dengan nama polutan itu sendiri (mis. kolom `NO2` untuk `jabon_NO2.csv`).

In [2]:
POLLUTANTS = {
    'CO':  '../data/downloads/jabon_pollutants_data/jabon_CO.csv',
    'NO2': '../data/downloads/jabon_pollutants_data/jabon_NO2.csv',
    'SO2': '../data/downloads/jabon_pollutants_data/jabon_SO2.csv',
}

fs = 1  # 1 sampel/hari

# ---------- Daftar PERSIS 68 Fitur yang Diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))
print("Polutan yang akan diproses:", list(POLLUTANTS.keys()))


Jumlah fitur yang diminta: 68
Polutan yang akan diproses: ['CO', 'NO2', 'SO2']


## 4. Fungsi Bantu

* `clean_series` — memuat & membersihkan data satu polutan (paksa numerik, buang outlier via IQR, imputasi linear berbasis waktu).
* `to_scalar` / `extract_one` — mengekstrak nilai skalar dari setiap fungsi fitur TSFEL.

In [3]:
def clean_series(csv_path, pollutant_col):
    """Muat dan bersihkan data deret waktu satu polutan."""
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # Paksa kolom target menjadi numerik
    df[pollutant_col] = pd.to_numeric(df[pollutant_col], errors='coerce')
    n_missing_before = df[pollutant_col].isna().sum()
    print(f"[{pollutant_col}] Jumlah nilai non-numerik/kosong awal: {n_missing_before}")

    # Deteksi dan kosongkan outlier (metode IQR)
    Q1 = df[pollutant_col].quantile(0.25)
    Q3 = df[pollutant_col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df.loc[(df[pollutant_col] < lower_bound) | (df[pollutant_col] > upper_bound), pollutant_col] = np.nan

    # Imputasi linear berbasis waktu
    df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

    return df_clean[pollutant_col].astype(float).values


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


def extract_features(signal, fs, feature_list=FEATURE_LIST):
    row = {}
    for fn_name in feature_list:
        row[fn_name] = extract_one(fn_name, signal, fs)
    return row


## 5. Eksekusi Ekstraksi Fitur untuk CO, NO₂, dan SO₂

In [4]:
results = {}

for pollutant, csv_path in POLLUTANTS.items():
    print(f"\n=== Memproses {pollutant} ===")
    signal_1d = clean_series(csv_path, pollutant)

    row = extract_features(signal_1d, fs)
    feat_df = pd.DataFrame([row])
    results[pollutant] = feat_df

    print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {pollutant}: {feat_df.shape[1]}")

    output_csv = f'{pollutant}_Jabon_TSFEL.csv'
    feat_df.to_csv(output_csv, index=False)
    print(f"Hasil ekstraksi tersimpan di: {output_csv}")



=== Memproses CO ===
[CO] Jumlah nilai non-numerik/kosong awal: 44
Berhasil! Jumlah fitur yang dihasilkan untuk CO: 68
Hasil ekstraksi tersimpan di: CO_Jabon_TSFEL.csv

=== Memproses NO2 ===
[NO2] Jumlah nilai non-numerik/kosong awal: 48
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68
Hasil ekstraksi tersimpan di: NO2_Jabon_TSFEL.csv

=== Memproses SO2 ===
[SO2] Jumlah nilai non-numerik/kosong awal: 40
Berhasil! Jumlah fitur yang dihasilkan untuk SO2: 68
Hasil ekstraksi tersimpan di: SO2_Jabon_TSFEL.csv


## 6. Gabungkan Hasil Ketiga Polutan

Menggabungkan hasil ekstraksi CO, NO₂, dan SO₂ menjadi satu tabel (satu baris per polutan)
agar mudah dikumpulkan sebagai satu berkas ringkasan.

In [5]:
combined = pd.concat(results.values(), keys=results.keys())
combined.index = combined.index.droplevel(1)
combined.index.name = 'pollutant'
combined = combined.reset_index()

combined_csv = 'ALL_Jabon_TSFEL.csv'
combined.to_csv(combined_csv, index=False)
print(f"Ringkasan gabungan tersimpan di: {combined_csv}")

combined


Ringkasan gabungan tersimpan di: ALL_Jabon_TSFEL.csv


,pollutant,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,CO,3.151509e-01,10.625441,3.0,8.634271e-04,185.064721,0.038312,0.029119,0.029092,0.019709,...,0.126932,0.633874,2.173399e-05,0.756976,0.001716,0.008553,2.131936,0.008357,7.834686e-05,0.0
1,NO2,4.222891e-07,0.011825,2.0,1.156956e-09,182.821046,0.000059,0.000032,0.000033,0.000003,...,0.152497,0.543632,1.718506e-10,0.002554,0.000001,0.000014,2.169289,0.000014,2.064401e-10,0.0
2,SO2,2.066779e-05,0.061631,2.0,5.662408e-08,200.138231,0.000660,0.000133,0.000123,-0.000363,...,0.153343,0.273626,6.820996e-08,0.053665,0.000007,0.000269,2.164427,0.000268,7.655132e-08,85.0


## 7. Struktur Pengelompokan Domain

Ke-68 fitur hasil ekstraksi (untuk setiap polutan) dikelompokkan secara terstruktur ke dalam 3 domain utama:

* **4.3 Fitur Domain Statistical (21 Fitur)** — mengukur distribusi probabilitas, momen statistik, dan dispersi nilai.
* **4.4 Fitur Domain Temporal & Fractal (21 Fitur)** — mengukur ketergantungan waktu, autokorelasi, dinamika puncak, dan kompleksitas non-linear.
* **4.5 Fitur Domain Spectral (26 Fitur)** — mengukur kandungan frekuensi, koefisien spektrogram, transformasi Fourier, dan energi wavelet.

### Hasil Ekstraksi Fitur TSFEL per Polutan (CSV)
*   **CO_Jabon_TSFEL.csv** — 68 fitur TSFEL untuk Karbon Monoksida:  
    <a href="CO_Jabon_TSFEL.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh CO_Jabon_TSFEL.csv</a>
*   **NO2_Jabon_TSFEL.csv** — 68 fitur TSFEL untuk Nitrogen Dioksida:  
    <a href="NO2_Jabon_TSFEL.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh NO2_Jabon_TSFEL.csv</a>
*   **SO2_Jabon_TSFEL.csv** — 68 fitur TSFEL untuk Sulfur Dioksida:  
    <a href="SO2_Jabon_TSFEL.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh SO2_Jabon_TSFEL.csv</a>
*   **ALL_Jabon_TSFEL.csv** — Ringkasan gabungan ketiga polutan:  
    <a href="ALL_Jabon_TSFEL.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh ALL_Jabon_TSFEL.csv</a>